# 🎵 Music Plagiarism Detection - Hybrid Approach
## ML Week Hackathon (IT-Jim)

**Chunk-based + Global metrics**

In [27]:
!pip install librosa -q

In [28]:
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass
import re, warnings
warnings.filterwarnings('ignore')
print("✅ Ready!")

✅ Ready!


In [ ]:
@dataclass
class Config:
    sr: int = 22050
    use_beat_sync: bool = True
    use_oti: bool = True
    chunk_duration_sec: float = 8.0
    chunk_overlap: float = 0.5
    chunk_threshold: float = 0.60
    w_chunk: float = 0.50
    w_dtw: float = 0.25
    w_cos: float = 0.15
    w_mfcc: float = 0.10
    threshold: float = 0.50

cfg = Config()

In [30]:
# PATHS
DATA = Path('/Users/ulanagusar/Desktop/ML_week-2026/smp_dataset')
ORIG = DATA / 'original'
COMP = DATA / 'comparison'
CSV = DATA / 'song_pairs_balanced.csv'

In [31]:
# === CORE FUNCTIONS ===
def load_audio(p): return librosa.util.normalize(librosa.load(p, sr=22050, mono=True)[0])
def hpcp(y): return librosa.feature.chroma_cens(y=y, sr=22050)
def mfcc(y): m = librosa.feature.mfcc(y=y, sr=22050, n_mfcc=13); return (m - m.mean(1,keepdims=True))/(m.std(1,keepdims=True)+1e-8)
def bsync(f, y): _, b = librosa.beat.beat_track(y=y, sr=22050); return librosa.util.sync(f, b) if len(b)>2 else f

def features(p, cfg):
    y = load_audio(p)
    h = hpcp(y)
    if cfg.use_beat_sync: h = bsync(h, y)
    m = mfcc(y)
    if cfg.use_beat_sync: m = bsync(m, y)
    return {'hpcp': h, 'mfcc': m}

def oti(h1, h2):
    g1, g2 = np.mean(h1,1), np.mean(h2,1)
    g1, g2 = g1/(np.linalg.norm(g1)+1e-8), g2/(np.linalg.norm(g2)+1e-8)
    return int(np.argmax(np.real(np.fft.ifft(np.fft.fft(g1)*np.conj(np.fft.fft(g2))))))

def apply_oti(h, o): return np.roll(h, o, axis=0) if o else h

In [32]:
# === CHUNK ANALYSIS ===
def segment(f, cf, hf):
    n = f.shape[1]; chunks = []; s = 0
    while s + cf <= n: chunks.append(f[:, s:s+cf]); s += hf
    if s < n and n-s >= cf*0.5: chunks.append(f[:, s:])
    return chunks

def cmp_chunk(c1, c2):
    l1, l2 = c1.shape[1], c2.shape[1]
    if l1 != l2:
        ml = max(l1,l2)
        if l1<ml: c1 = np.pad(c1,((0,0),(0,ml-l1)),mode='edge')
        if l2<ml: c2 = np.pad(c2,((0,0),(0,ml-l2)),mode='edge')
    try: D,_ = librosa.sequence.dtw(c1, c2, metric='cosine'); dtw = max(0, 1-D[-1,-1]/(D.shape[0]+D.shape[1]))
    except: dtw = 0
    g1, g2 = np.mean(c1,1), np.mean(c2,1)
    cos = max(0, np.dot(g1,g2)/(np.linalg.norm(g1)*np.linalg.norm(g2)+1e-8))
    return 0.6*dtw + 0.4*cos

def chunk_analysis(qh, rh, cfg):
    fps = 2 if qh.shape[1]<100 else 43
    cf = max(5, int(cfg.chunk_duration_sec * fps))
    hf = max(2, int(cf * (1 - cfg.chunk_overlap)))
    qc, rc = segment(qh, cf, hf), segment(rh, cf, hf)
    nq, nr = len(qc), len(rc)
    if nq==0 or nr==0: return {'score':0, 'n_plag':0, 'n_total':0, 'matches':[], 'matrix':np.array([[]])}
    
    mat = np.array([[cmp_chunk(qc[i], rc[j]) for j in range(nr)] for i in range(nq)])
    matches = [(int(np.argmax(mat[i])), float(mat[i].max())) for i in range(nq)]
    n_plag = sum(1 for _,s in matches if s >= cfg.chunk_threshold)
    sims = [s for _,s in matches]
    
    # consecutive bonus
    mx_con, cur = 0, 0
    for _,s in matches:
        if s >= cfg.chunk_threshold: cur += 1; mx_con = max(mx_con, cur)
        else: cur = 0
    bonus = min(0.15, (mx_con-1)*0.05) if mx_con>1 else 0
    
    score = min(1.0, 0.4*np.mean(sims) + 0.35*(n_plag/nq) + 0.25*np.max(sims) + bonus)
    return {'score': score, 'n_plag': n_plag, 'n_total': nq, 'matches': matches, 'matrix': mat}

In [33]:
# === GLOBAL METRICS ===
def g_dtw(f1,f2):
    try: D,_ = librosa.sequence.dtw(f1,f2,metric='cosine'); return max(0,1-D[-1,-1]/(D.shape[0]+D.shape[1]))
    except: return 0
def g_cos(f1,f2): g1,g2=np.mean(f1,1),np.mean(f2,1); return max(0,np.dot(g1,g2)/(np.linalg.norm(g1)*np.linalg.norm(g2)+1e-8))
def g_mfcc(m1,m2):
    try: D,_=librosa.sequence.dtw(m1,m2,metric='euclidean'); return max(0,1-D[-1,-1]/(D.shape[0]+D.shape[1])/10)
    except: return 0

In [34]:
# === MAIN COMPARE ===
def compare(query_path, ref_path, cfg):
    qf, rf = features(query_path, cfg), features(ref_path, cfg)
    o = oti(qf['hpcp'], rf['hpcp']) if cfg.use_oti else 0
    rh = apply_oti(rf['hpcp'], o)
    
    ca = chunk_analysis(qf['hpcp'], rh, cfg)
    dtw = g_dtw(qf['hpcp'], rh)
    cos = g_cos(qf['hpcp'], rh)
    mf = g_mfcc(qf['mfcc'], rf['mfcc'])
    
    score = cfg.w_chunk*ca['score'] + cfg.w_dtw*dtw + cfg.w_cos*cos + cfg.w_mfcc*mf
    if ca['n_plag']/ca['n_total'] > 0.5 and ca['matches'] and max(s for _,s in ca['matches']) > 0.7:
        score = max(score, ca['score']*0.95)
    
    return {'score': score, 'is_plag': score >= cfg.threshold, 'chunk': ca['score'],
            'n_plag': ca['n_plag'], 'n_total': ca['n_total'], 'plag_pct': ca['n_plag']/ca['n_total']*100,
            'dtw': dtw, 'cos': cos, 'mfcc': mf, 'oti': o, 'matrix': ca['matrix'], 'matches': ca['matches']}

In [35]:
# === FILE FINDER ===
def find(folder, title):
    ts = re.sub(r'[^a-z0-9]', '', title.lower())
    for f in Path(folder).glob('*'):
        if f.is_file():
            fs = re.sub(r'[^a-z0-9]', '', f.stem.lower())
            if fs==ts or (len(ts)>10 and ts[:25] in fs) or (len(fs)>10 and fs[:25] in ts): return f
    return None

In [36]:
# === LOAD DATA ===
df = pd.read_csv(CSV)
print(f"Total: {len(df)}")
print(df['relation'].value_counts())

Total: 32
relation
not_plag      16
plag           6
remake         5
plag_doubt     5
Name: count, dtype: int64


In [37]:
# === TEST ===
row = df.iloc[4]
ori, comp = find(ORIG, row['ori_title']), find(COMP, row['comp_title'])
if ori and comp:
    res = compare(str(comp), str(ori), cfg)
    print("🚨 PLAG" if res['is_plag'] else "✅ NO")
    print(f"Score: {res['score']:.3f} | Chunks: {res['n_plag']}/{res['n_total']} | DTW: {res['dtw']:.3f}")

🚨 PLAG
Score: 0.950 | Chunks: 32/32 | DTW: 0.890


In [38]:
# === EVALUATE ALL ===
results = []
for idx, row in df.iterrows():
    true = 0 if row['relation']=='not_plag' else 1
    ori, comp = find(ORIG, row['ori_title']), find(COMP, row['comp_title'])
    if not ori or not comp: print(f"[{idx}] ⚠️"); results.append({'true':true,'pred':None,'score':None}); continue
    try:
        r = compare(str(comp), str(ori), cfg)
        pred = 1 if r['is_plag'] else 0
        print(f"[{idx}] {'✓' if pred==true else '✗'} {r['score']:.3f} | {r['n_plag']}/{r['n_total']} | {row['relation']}")
        results.append({'true':true, 'pred':pred, 'score':r['score'], 'chunk':r['chunk']})
    except Exception as e: print(f"[{idx}] ❌"); results.append({'true':true,'pred':None,'score':None})

rdf = pd.DataFrame(results)

[0] ✓ 0.972 | 31/31 | plag
[1] ✓ 0.958 | 33/33 | remake
[2] ✓ 0.957 | 15/15 | remake
[3] ✓ 0.956 | 28/28 | plag
[4] ✓ 0.950 | 32/32 | plag
[5] ✓ 0.950 | 35/35 | plag
[6] ✓ 0.958 | 19/19 | plag_doubt
[7] ✓ 0.965 | 22/22 | plag_doubt
[8] ✓ 0.950 | 43/43 | remake
[9] ✓ 0.910 | 1/1 | plag_doubt
[10] ✓ 0.963 | 23/23 | plag_doubt
[11] ✓ 0.950 | 37/37 | plag_doubt
[12] ✓ 0.961 | 1/1 | remake
[13] ✓ 0.950 | 9/9 | remake
[14] ✓ 0.958 | 46/46 | plag
[15] ✓ 0.955 | 1/1 | plag
[16] ✗ 0.950 | 35/35 | not_plag
[17] ✗ 0.960 | 32/32 | not_plag
[18] ✗ 0.950 | 43/43 | not_plag
[19] ❌
[20] ✗ 0.950 | 31/31 | not_plag
[21] ✗ 0.950 | 23/23 | not_plag
[22] ✗ 0.950 | 32/32 | not_plag
[23] ✗ 0.950 | 15/15 | not_plag
[24] ✗ 0.960 | 19/19 | not_plag
[25] ✗ 0.965 | 28/28 | not_plag
[26] ✗ 0.950 | 33/33 | not_plag
[27] ✗ 0.955 | 9/9 | not_plag
[28] ✗ 0.954 | 22/22 | not_plag
[29] ✗ 0.950 | 1/1 | not_plag
[30] ✗ 0.952 | 46/46 | not_plag
[31] ✗ 0.950 | 37/37 | not_plag


In [39]:
# === METRICS ===
v = rdf.dropna()
if len(v):
    yt, yp = v['true'].values, v['pred'].values
    tp,fp,fn,tn = ((yt==1)&(yp==1)).sum(), ((yt==0)&(yp==1)).sum(), ((yt==1)&(yp==0)).sum(), ((yt==0)&(yp==0)).sum()
    print(f"\nAcc: {(tp+tn)/(tp+tn+fp+fn):.3f}")
    print(f"F1: {2*tp/(2*tp+fp+fn):.3f}" if 2*tp+fp+fn else "F1: 0")
    print(f"Confusion: TP={tp} FP={fp} FN={fn} TN={tn}")


Acc: 0.516
F1: 0.681
Confusion: TP=16 FP=15 FN=0 TN=0
